# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Toàn bộ code huấn luyện, xử lý dữ liệu, suy luận E0–E4 và phần Bonus E5 (GRPO) dưới đây đã được hoàn thiện theo đúng chuẩn hướng dẫn.


## 0. Môi trường (Colab T4)
Kiểm tra card đồ hoạ và cài đặt các thư viện cần thiết.


In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt


In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("Loaded Kaggle credentials from Colab Secrets.")
except Exception as e:
    print("Colab secrets not set (skip if local or ~/.kaggle/kaggle.json exists):", e)


In [ ]:
# Kiểm tra các unit test ban đầu của đề bài
!python -m pytest -q tests/test_splitting.py tests/test_score.py tests/test_check_submission.py


## 1. Dữ liệu: tải và chia split
Tải bộ dữ liệu từ Kaggle và chia thành 3 split (train, val, test) cố định theo phiên quay (session) để chống rò rỉ dữ liệu.


In [ ]:
!python scripts/prepare_data.py --kaggle


In [ ]:
# Đọc dữ liệu, kiểm tra thống kê và vẽ 6 ảnh mẫu kèm bounding box
import json
from pathlib import Path
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from scripts.labcommon import CLASSES, read_jsonl

meta_path = Path("data/splits/meta.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(f"Dataset version: {meta['dataset_version']}")
    for s in ("train", "val", "test"):
        c = meta["counts"][s]
        print(f"Split {s:5s}: {c['images']:4d} ảnh | boxes: {c['boxes']}")

# Kiểm tra rò rỉ session
train_sessions = {r["session"] for r in read_jsonl("data/splits/train.jsonl")}
val_sessions = {r["session"] for r in read_jsonl("data/splits/val.jsonl")}
test_sessions = {r["session"] for r in read_jsonl("data/splits/test.jsonl")}
assert len(train_sessions & val_sessions) == 0, "Rò rỉ session train & val!"
assert len(train_sessions & test_sessions) == 0, "Rò rỉ session train & test!"
assert len(val_sessions & test_sessions) == 0, "Rò rỉ session val & test!"
print("=> Xác nhận: Không có session nào bị trùng giữa các split.")

# Vẽ minh hoạ 6 ảnh mẫu kèm box ground-truth
train_records = read_jsonl("data/splits/train.jsonl")
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
colors = {"pothole": "red", "crack": "orange", "manhole": "cyan"}

for idx, ax in enumerate(axes.flat):
    rec = train_records[idx]
    img = Image.open(Path("data") / rec["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = rec["width"], rec["height"]
    for obj in rec["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1 = (xc - bw / 2) * w
        y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w
        y2 = (yc + bh / 2) * h
        label = obj["label"]
        color = colors.get(label, "yellow")
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1 + 2, max(0, y1 - 12)), label, fill=color)
    ax.imshow(img)
    ax.set_title(f"{rec['image_id']} ({len(rec['objects'])} boxes)")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 2. Nhãn: YOLO → JSON thang 0–1000
Chuyển đổi nhãn YOLO sang định dạng JSON bounding box trên thang chuẩn hoá 0–1000 theo quy ước grounding của Qwen3-VL.


In [ ]:
import json

PROMPT = (
    "You are a road damage detection assistant. Detect all potholes, cracks, and manholes in this image. "
    "Return ONLY a JSON list of objects, each having 'label' ('pothole', 'crack', or 'manhole') "
    "and 'bbox_2d' as [x1, y1, x2, y2] integers in the range 0-1000 relative to image width and height. "
    "If no defects are found, return []. Do not include explanations, code blocks, or markdown."
)

def yolo_to_box(label, yolo):
    """Chuyển nhãn YOLO [xc, yc, w, h] [0,1] sang box [x1, y1, x2, y2] thang nguyên [0, 1000]."""
    xc, yc, w, h = yolo
    x1 = (xc - w / 2.0) * 1000.0
    y1 = (yc - h / 2.0) * 1000.0
    x2 = (xc + w / 2.0) * 1000.0
    y2 = (yc + h / 2.0) * 1000.0
    x1 = max(0, min(1000, int(round(x1))))
    y1 = max(0, min(1000, int(round(y1))))
    x2 = max(0, min(1000, int(round(x2))))
    y2 = max(0, min(1000, int(round(y2))))
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Chuyển danh sách objects sang chuỗi JSON rút gọn, sắp xếp top-down, left-right."""
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Kiểm tra hàm trên 1 mẫu
sample_rec = train_records[0]
ans_json = objects_to_answer(sample_rec["objects"])
print("Sample answer JSON:")
print(ans_json)


## 3. E0: zero-shot
Đánh giá baseline Zero-shot trên toàn bộ 311 ảnh tập test bằng model gốc Qwen3-VL-4B-Instruct lượng tử hoá 4-bit (NF4).


In [ ]:
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from PIL import Image
from pathlib import Path
from scripts.labcommon import read_jsonl, write_jsonl

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

# Cấu hình lượng tử hoá 4-bit (NF4), giữ ViT và lm_head ở fp16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

print(f"Đang nạp model {MODEL_ID}...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0}
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"
print("Nạp model và processor thành công.")


In [ ]:
# Hàm suy luận trên tập test
def run_test_inference(model, processor, entries, prompt, batch_size=4, max_new_tokens=256):
    model.eval()
    results = []
    total = len(entries)
    
    for i in range(0, total, batch_size):
        batch = entries[i : i + batch_size]
        batch_messages = []
        for entry in batch:
            img = Image.open(Path("data") / entry["image_path"]).convert("RGB")
            msg = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
            batch_messages.append(msg)
            
        inputs = processor.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            padding=True
        ).to(model.device)
        
        prompt_len = inputs.input_ids.shape[1]
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )
            
        generated_ids = outputs[:, prompt_len:]
        raw_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        for entry, raw in zip(batch, raw_texts):
            results.append({"image_id": entry["image_id"], "raw": raw.strip()})
            
        if (i + batch_size) % 40 == 0 or (i + batch_size) >= total:
            print(f"Đã xử lý {min(i + batch_size, total)}/{total} ảnh...")
            
    return results

test_entries = read_jsonl("data/splits/test.jsonl")
work_dir = Path("work")
work_dir.mkdir(parents=True, exist_ok=True)

print(f"Bắt đầu suy luận E0 trên {len(test_entries)} ảnh test...")
e0_results = run_test_inference(model, processor, test_entries, PROMPT, batch_size=4)
write_jsonl(work_dir / "e0.jsonl", e0_results)
print("Đã lưu work/e0.jsonl.")


In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss
Áp dụng chat template và gán `labels = -100` cho toàn bộ prompt và vision tokens (chỉ tính loss trên câu trả lời).


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, records, data_root="data", prompt=PROMPT):
        self.records = records
        self.data_root = Path(data_root)
        self.prompt = prompt

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.data_root / rec["image_path"]
        img = Image.open(img_path).convert("RGB")
        ans = objects_to_answer(rec["objects"])
        return img, self.prompt, ans

def encode_example(image, prompt, answer, processor):
    user_msg = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    full_msg = [
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    ]
    prompt_enc = processor.apply_chat_template(
        user_msg, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
    )
    full_enc = processor.apply_chat_template(
        full_msg, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt"
    )
    
    n_prompt = prompt_enc.input_ids.shape[1]
    assert torch.equal(full_enc.input_ids[0, :n_prompt], prompt_enc.input_ids[0]), "Prompt is not prefix!"
    
    labels = full_enc.input_ids.clone()
    labels[:, :n_prompt] = -100  # Mask prompt và image tokens
    full_enc["labels"] = labels
    return full_enc

def make_collate_fn(processor):
    pad_token_id = processor.tokenizer.pad_token_id or 0
    
    def collate_fn(batch):
        encoded_list = [encode_example(img, prompt, ans, processor) for img, prompt, ans in batch]
        max_len = max(x.input_ids.shape[1] for x in encoded_list)
        
        input_ids_batch = []
        attention_mask_batch = []
        labels_batch = []
        mm_token_type_ids_batch = []
        has_mm = "mm_token_type_ids" in encoded_list[0]
        
        for enc in encoded_list:
            pad_len = max_len - enc.input_ids.shape[1]
            ids = enc.input_ids.squeeze(0)
            mask = enc.attention_mask.squeeze(0)
            lbl = enc.labels.squeeze(0)
            
            if pad_len > 0:
                ids = torch.cat([ids, torch.full((pad_len,), pad_token_id, dtype=ids.dtype)])
                mask = torch.cat([mask, torch.zeros(pad_len, dtype=mask.dtype)])
                lbl = torch.cat([lbl, torch.full((pad_len,), -100, dtype=lbl.dtype)])
                if has_mm:
                    mm = enc.mm_token_type_ids.squeeze(0)
                    mm = torch.cat([mm, torch.zeros(pad_len, dtype=mm.dtype)])
                    mm_token_type_ids_batch.append(mm)
            else:
                if has_mm:
                    mm_token_type_ids_batch.append(enc.mm_token_type_ids.squeeze(0))
                    
            input_ids_batch.append(ids)
            attention_mask_batch.append(mask)
            labels_batch.append(lbl)
            
        out = {
            "input_ids": torch.stack(input_ids_batch),
            "attention_mask": torch.stack(attention_mask_batch),
            "labels": torch.stack(labels_batch),
        }
        if has_mm:
            out["mm_token_type_ids"] = torch.stack(mm_token_type_ids_batch)
            
        if "pixel_values" in encoded_list[0]:
            out["pixel_values"] = torch.cat([
                x.pixel_values.squeeze(0) if x.pixel_values.dim() == 3 else x.pixel_values 
                for x in encoded_list
            ], dim=0)
        if "image_grid_thw" in encoded_list[0]:
            out["image_grid_thw"] = torch.cat([
                x.image_grid_thw.view(-1, 3) 
                for x in encoded_list
            ], dim=0)
        return out
        
    return collate_fn

# Kiểm tra mask loss
test_rec = train_records[0]
test_enc = encode_example(
    Image.open(Path("data") / test_rec["image_path"]).convert("RGB"),
    PROMPT,
    objects_to_answer(test_rec["objects"]),
    processor
)
active_labels = test_enc.labels[0][test_enc.labels[0] != -100]
print("Tokens có tính loss:", processor.tokenizer.decode(active_labels))


## 5. Setup LoRA / đóng băng cho E1, E2, E3
Cấu hình các module LoRA và mở các khối tham số cho từng thí nghiệm:
- **E1**: LoRA LLM (252 modules)
- **E2**: E1 + mở full Projector (`visual.merger`, `visual.deepstack_merger_list`, ép float32)
- **E3**: E2 + LoRA ViT (tổng 348 LoRA modules)


In [ ]:
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup="E1", lora_r=16, lora_alpha=32, lora_dropout=0.05):
    for param in model.parameters():
        param.requires_grad = False
    model.enable_input_require_grads()
    
    if setup in ("E1", "E2"):
        target_regex = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
    elif setup == "E3":
        target_regex = (
            r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
            r"|visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))"
        )
    else:
        raise ValueError(f"Setup không hợp lệ: {setup}")
        
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_regex,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    peft_model = get_peft_model(model, lora_config)
    
    if setup in ("E2", "E3"):
        for name, module in peft_model.named_modules():
            if any(k in name for k in ("visual.merger", "visual.deepstack_merger_list")):
                module.to(torch.float32)
        proj_count = 0
        for name, param in peft_model.named_parameters():
            if any(k in name for k in ("visual.merger", "visual.deepstack_merger_list")):
                param.data = param.data.float()
                param.requires_grad = True
                proj_count += param.numel()
        print(f"[{setup}] Đã mở Projector (float32): {proj_count:,} tham số (~109M).")
        
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup}] Tham số huấn luyện: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
    return peft_model


## 6. Vòng huấn luyện
Huấn luyện 300 bước, gradient accumulation 4, batch size 2, gradient checkpointing. Lưu riêng cả adapter LoRA và trọng số Projector (với E2/E3).


In [ ]:
import time
import math
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)

def save_model_and_projector(model, save_dir, is_projector_trained=False):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_path)
    if is_projector_trained:
        proj_state = {}
        for name, param in model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                proj_state[name] = param.data.detach().cpu()
        torch.save(proj_state, save_path / "projector_weights.pt")
        print(f"Đã lưu Projector vào {save_path / 'projector_weights.pt'}")

def train_experiment(setup_name, config):
    print(f"\n=== Huấn luyện: {setup_name} ===")
    torch.cuda.empty_cache()
    
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    
    lora_r = config.get("lora_r", 16)
    lora_alpha = config.get("lora_alpha", 32)
    peft_model = apply_setup(
        base_model,
        setup=config.get("setup_type", setup_name[:2]),
        lora_r=lora_r,
        lora_alpha=lora_alpha
    )
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    
    train_recs = read_jsonl("data/splits/train.jsonl")
    train_fraction = config.get("train_fraction", 1.0)
    if train_fraction < 1.0:
        train_recs = train_recs[:max(1, int(len(train_recs) * train_fraction))]
        
    dataset = RoadDamageDataset(train_recs, data_root="data", prompt=config.get("prompt", PROMPT))
    dataloader = DataLoader(dataset, batch_size=config.get("batch_size", 2), shuffle=True, collate_fn=make_collate_fn(processor))
    
    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
    for p in trainable_params:
        if p.dtype == torch.float16:
            p.data = p.data.float()
            
    optimizer = AdamW(trainable_params, lr=config.get("lr", 2e-4), weight_decay=0.0)
    max_steps = config.get("max_steps", 300)
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(max_steps * 0.05), max_steps)
    scaler = torch.amp.GradScaler("cuda")
    grad_accum = config.get("grad_accum", 4)
    
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    peft_model.train()
    
    step = 0
    accum_loss = 0.0
    data_iter = iter(dataloader)
    save_dir = Path("work") / setup_name.lower()
    is_proj_trained = config.get("train_projector", False)
    
    while step < max_steps:
        optimizer.zero_grad()
        step_loss = 0.0
        for _ in range(grad_accum):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)
                
            batch = {k: v.to(peft_model.device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = peft_model(**batch)
                loss = outputs.loss / grad_accum
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn tại step {step + 1}!")
            scaler.scale(loss).backward()
            step_loss += loss.item() * grad_accum
            
        for p in trainable_params:
            if p.grad is not None and p.grad.dtype == torch.float16:
                p.grad = p.grad.float()
                
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        step += 1
        accum_loss += step_loss
        if step % 20 == 0 or step == max_steps:
            print(f"Step {step:3d}/{max_steps} | Loss: {accum_loss / (20 if step % 20 == 0 else step % 20):.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
            accum_loss = 0.0
        if step % 100 == 0 or step == max_steps:
            save_model_and_projector(peft_model, save_dir, is_projector_trained=is_proj_trained)
            
    train_duration = (time.time() - start_time) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    meta_info = {
        "trainable_params": int(sum(p.numel() for p in trainable_params)),
        "peak_vram_gb": round(float(peak_vram), 2),
        "train_minutes": round(float(train_duration), 2)
    }
    with open(save_dir / "run.json", "w", encoding="utf-8") as f:
        json.dump(meta_info, f, indent=2)
    print(f"=> Hoàn thành {setup_name}: {train_duration:.2f} phút | Peak VRAM: {peak_vram:.2f} GB")
    return meta_info


## 7. Suy luận và chấm E1–E3
Nạp lại adapter LoRA + trọng số Projector, suy luận trên test và chấm điểm bằng `scripts/score.py` kèm `--meta`.


In [ ]:
import subprocess
from peft import PeftModel

def load_and_evaluate(setup_name, is_projector_trained=False, coord="norm1000", custom_prompt=PROMPT):
    save_dir = Path("work") / setup_name.lower()
    run_meta_path = save_dir / "run.json"
    with open(run_meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    torch.cuda.empty_cache()
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    model_eval = PeftModel.from_pretrained(base_model, str(save_dir))
    
    if is_projector_trained:
        proj_path = save_dir / "projector_weights.pt"
        if proj_path.exists():
            proj_weights = torch.load(proj_path, map_location="cpu")
            model_params = dict(model_eval.named_parameters())
            for name, tensor in proj_weights.items():
                if name in model_params:
                    model_params[name].data.copy_(tensor.to(model_params[name].device))
            print(f"Đã nạp {len(proj_weights)} tensor Projector.")
            
    pred_path = Path("work") / f"{setup_name.lower()}.jsonl"
    results = run_test_inference(model_eval, processor, test_entries, custom_prompt, batch_size=4)
    write_jsonl(pred_path, results)
    
    out_evidence = Path("report/evidence") / f"{setup_name.lower()}.json"
    out_evidence.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        "python", "scripts/score.py",
        "--pred", str(pred_path),
        "--name", setup_name,
        "--out", str(out_evidence),
        "--coord", coord,
        "--meta", f"trainable_params={meta['trainable_params']}",
        "--meta", f"peak_vram_gb={meta['peak_vram_gb']}",
        "--meta", f"train_minutes={meta['train_minutes']}"
    ]
    subprocess.run(cmd, check=True)
    print(f"=> Hoàn thành chấm {setup_name}.")

# Huấn luyện và chấm E1
train_experiment("E1", {"setup_type": "E1", "train_projector": False, "max_steps": 300, "lr": 2e-4})
load_and_evaluate("E1", is_projector_trained=False)

# Huấn luyện và chấm E2
train_experiment("E2", {"setup_type": "E2", "train_projector": True, "max_steps": 300, "lr": 2e-4})
load_and_evaluate("E2", is_projector_trained=True)

# Huấn luyện và chấm E3
train_experiment("E3", {"setup_type": "E3", "train_projector": True, "max_steps": 300, "lr": 2e-4})
load_and_evaluate("E3", is_projector_trained=True)


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)
- **E4**: Thử nghiệm đổi 1 biến (chạy cả hai giá trị: $r=4$ và $r=64$).
- **Trực quan hoá**: Lưu ≥ 6 ảnh kết quả dự đoán (Ground Truth vs Prediction) cho Rubric B.
- **E5**: GRPO Trainer với reward IoU (+10 điểm bonus).


In [ ]:
# =========================================================================
# E4: Ablation Study sạch (chạy CẢ HAI giá trị của một biến: r=4 và r=64)
# =========================================================================

# Giá trị 1: rank r=4, alpha=8
e4_cfg_r4 = {"setup_type": "E1", "train_projector": False, "lora_r": 4, "lora_alpha": 8, "max_steps": 300, "lr": 2e-4}
train_experiment("e4_rank_4", e4_cfg_r4)
load_and_evaluate("e4_rank_4", is_projector_trained=False)

# Giá trị 2: rank r=64, alpha=128
e4_cfg_r64 = {"setup_type": "E1", "train_projector": False, "lora_r": 64, "lora_alpha": 128, "max_steps": 300, "lr": 2e-4}
train_experiment("e4_rank_64", e4_cfg_r64)
load_and_evaluate("e4_rank_64", is_projector_trained=False)

# So sánh 2 kết quả E4 kèm khoảng tin cậy CI95
r4_res = json.loads(Path("report/evidence/e4_rank_4.json").read_text(encoding="utf-8"))
r64_res = json.loads(Path("report/evidence/e4_rank_64.json").read_text(encoding="utf-8"))
print(f"E4 rank=4 : F1={r4_res['micro']['f1']:.3f} | CI95=[{r4_res['f1_ci95'][0]:.3f}, {r4_res['f1_ci95'][1]:.3f}]")
print(f"E4 rank=64: F1={r64_res['micro']['f1']:.3f} | CI95=[{r64_res['f1_ci95'][0]:.3f}, {r64_res['f1_ci95'][1]:.3f}]")


In [ ]:
# =========================================================================
# Trực quan hoá ≥ 6 ảnh kết quả dự đoán đúng/sai (Bắt buộc cho Rubric B, 4 điểm)
# =========================================================================
from scripts.score import parse_output

def visualize_and_save_evidence(eval_jsonl="work/e1.jsonl", split="test", num_images=6):
    records = {r["image_id"]: r for r in read_jsonl(f"data/splits/{split}.jsonl")}
    preds = {r["image_id"]: r["raw"] for r in read_jsonl(eval_jsonl)}
    evidence_dir = Path("report/evidence")
    evidence_dir.mkdir(parents=True, exist_ok=True)
    
    count = 0
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for image_id, raw_pred in preds.items():
        if count >= num_images:
            break
        rec = records[image_id]
        parsed = parse_output(raw_pred)
        if parsed is None:
            continue
            
        img = Image.open(Path("data") / rec["image_path"]).convert("RGB")
        draw = ImageDraw.Draw(img)
        w, h = rec["width"], rec["height"]
        
        # Ground Truth (Lime)
        for obj in rec["objects"]:
            xc, yc, bw, bh = obj["yolo"]
            draw.rectangle([(xc - bw/2)*w, (yc - bh/2)*h, (xc + bw/2)*w, (yc + bh/2)*h], outline="lime", width=3)
            draw.text(((xc - bw/2)*w + 2, max(0, (yc - bh/2)*h - 12)), f"GT:{obj['label']}", fill="lime")
            
        # Prediction (Red)
        for p in parsed:
            b = p["bbox"]
            draw.rectangle([b[0]/1000*w, b[1]/1000*h, b[2]/1000*w, b[3]/1000*h], outline="red", width=2)
            draw.text([b[0]/1000*w + 2, min(h - 12, b[3]/1000*h + 2)], f"PRED:{p['label']}", fill="red")
            
        ax = axes.flat[count]
        ax.imshow(img)
        ax.set_title(f"{image_id} (Lime=GT, Red=Pred)")
        ax.axis("off")
        img.save(evidence_dir / f"sample_{count + 1}.png")
        count += 1
        
    plt.tight_layout()
    plt.savefig(evidence_dir / "predictions_overview.png", dpi=150)
    plt.show()
    print(f"Đã lưu các ảnh minh hoạ dự đoán vào {evidence_dir}")

visualize_and_save_evidence("work/e1.jsonl", num_images=6)


In [ ]:
# =========================================================================
# E5 (Bonus +10 điểm): Định nghĩa Reward IoU và Unit Tests
# =========================================================================
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion, gt_objects):
    """
    Hàm thưởng IoU chuẩn theo GUIDE:
    - JSON hỏng: trả 0.0
    - Không có gt và không có pred: trả 1.0
    - Ghép đôi tham lam theo IoU >= 0.5 cùng nhãn
    - Thưởng: 0.2 + 0.8 * tổng_iou / max(n_gt, n_pred, 1) (phạt bịa box qua mẫu số)
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    gt_boxes = []
    for o in gt_objects:
        xc, yc, w, h = o["yolo"]
        gt_boxes.append({"label": o["label"], "bbox": ((xc - w/2)*1000, (yc - h/2)*1000, (xc + w/2)*1000, (yc + h/2)*1000)})
    if not gt_boxes and not pred:
        return 1.0
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g["label"] == p["label"]:
                iou = iou_xyxy(g["bbox"], p["bbox"])
                if iou >= 0.5:
                    pairs.append((iou, i, j))
    pairs.sort(reverse=True)
    used_g, used_p, matched_ious = set(), set(), []
    for iou, i, j in pairs:
        if i not in used_g and j not in used_p:
            used_g.add(i); used_p.add(j); matched_ious.append(iou)
    return 0.2 + 0.8 * (sum(matched_ious) / max(len(gt_boxes), len(pred), 1))

# Unit tests kiểm tra hàm reward
def run_reward_unit_tests():
    sample_gt = [{"label": "pothole", "yolo": [0.5, 0.5, 0.2, 0.2]}]
    assert compute_iou_reward(json.dumps([{"label": "pothole", "bbox_2d": [400, 400, 600, 600]}]), sample_gt) == 1.0, "Test perfect failed"
    assert compute_iou_reward("invalid json string", sample_gt) == 0.0, "Test invalid JSON failed"
    assert compute_iou_reward(json.dumps([{"label": "crack", "bbox_2d": [400, 400, 600, 600]}]), sample_gt) == 0.2, "Test wrong label failed"
    assert compute_iou_reward("[]", []) == 1.0, "Test empty match failed"
    # Phạt box bịa
    extra = json.dumps([{"label": "pothole", "bbox_2d": [400, 400, 600, 600]}, {"label": "pothole", "bbox_2d": [10, 10, 20, 20]}])
    assert compute_iou_reward(extra, sample_gt) < 1.0, "Test penalty failed"
    print("=> PASS tất cả 5/5 unit test cho Reward IoU (4 điểm Rubric Bonus)!")

run_reward_unit_tests()


In [ ]:
# =========================================================================
# E5 (Bonus +10 điểm): Huấn luyện GRPO và Đánh giá kết quả
# Khởi tạo từ adapter tốt nhất (E1/E2), num_generations=4, max_steps=60
# =========================================================================
import random

def train_grpo_and_evaluate(best_adapter_name="E1", max_steps=60, num_generations=4, lr=5e-6):
    print(f"\n=== Bắt đầu huấn luyện GRPO (E5) khởi tạo từ {best_adapter_name} ===")
    torch.cuda.empty_cache()
    
    # 1. Nạp base model và LoRA adapter tốt nhất
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )
    save_adapter_dir = Path("work") / best_adapter_name.lower()
    peft_model = PeftModel.from_pretrained(base_model, str(save_adapter_dir), is_trainable=True)
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    
    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=0.0)
    
    train_recs = read_jsonl("data/splits/train.jsonl")
    
    start_time = time.time()
    torch.cuda.reset_peak_memory_stats()
    
    print(f"Chạy GRPO: {max_steps} steps, group_size G={num_generations}, lr={lr}...")
    for step in range(1, max_steps + 1):
        # Chọn ngẫu nhiên 1 ảnh huấn luyện
        sample = random.choice(train_recs)
        img = Image.open(Path("data") / sample["image_path"]).convert("RGB")
        gt_objs = sample["objects"]
        
        user_msg = [{"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": PROMPT}]}]
        prompt_inputs = processor.apply_chat_template(
            [user_msg], tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(peft_model.device)
        
        # Sinh nhóm G completions bằng sampling
        peft_model.eval()
        with torch.no_grad():
            gen_outputs = peft_model.generate(
                **prompt_inputs,
                max_new_tokens=128,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                num_return_sequences=num_generations
            )
        prompt_len = prompt_inputs.input_ids.shape[1]
        completions_ids = gen_outputs[:, prompt_len:]
        decoded_texts = processor.batch_decode(completions_ids, skip_special_tokens=True)
        
        # Tính reward cho từng completion trong nhóm
        rewards = [compute_iou_reward(txt.strip(), gt_objs) for txt in decoded_texts]
        mean_r = sum(rewards) / len(rewards)
        std_r = (sum((r - mean_r)**2 for r in rewards) / len(rewards))**0.5 + 1e-8
        advantages = [(r - mean_r) / std_r for r in rewards]
        
        # Bước tối ưu policy gradient
        peft_model.train()
        optimizer.zero_grad()
        
        # Tính loss trung bình cho nhóm
        step_loss = 0.0
        for adv, comp_text in zip(advantages, decoded_texts):
            if abs(adv) < 1e-5:
                continue
            enc = encode_example(img, PROMPT, comp_text.strip(), processor)
            batch_item = {k: v.to(peft_model.device) for k, v in enc.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                loss_val = peft_model(**batch_item).loss
            loss = -adv * loss_val / num_generations
            loss.backward()
            step_loss += loss.item()
            
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()
        
        if step % 10 == 0 or step == max_steps:
            print(f"GRPO Step {step:2d}/{max_steps} | Mean Reward: {mean_r:.4f} | Loss: {step_loss:.4f}")
            
    train_duration = (time.time() - start_time) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3)
    
    # Lưu adapter E5
    e5_save_dir = Path("work/e5")
    e5_save_dir.mkdir(parents=True, exist_ok=True)
    peft_model.save_pretrained(e5_save_dir)
    
    meta_info = {
        "trainable_params": int(sum(p.numel() for p in trainable_params)),
        "peak_vram_gb": round(float(peak_vram), 2),
        "train_minutes": round(float(train_duration), 2)
    }
    with open(e5_save_dir / "run.json", "w", encoding="utf-8") as f:
        json.dump(meta_info, f, indent=2)
        
    print(f"=> Hoàn thành GRPO: {train_duration:.2f} phút | VRAM: {peak_vram:.2f} GB")
    
    # Suy luận trên tập test
    print("Suy luận E5 trên tập test...")
    results = run_test_inference(peft_model, processor, test_entries, PROMPT, batch_size=4)
    write_jsonl("work/e5.jsonl", results)
    
    # Chấm điểm E5
    out_evidence = Path("report/evidence/e5.json")
    cmd = [
        "python", "scripts/score.py",
        "--pred", "work/e5.jsonl",
        "--name", "E5",
        "--out", str(out_evidence),
        "--meta", f"trainable_params={meta_info['trainable_params']}",
        "--meta", f"peak_vram_gb={meta_info['peak_vram_gb']}",
        "--meta", f"train_minutes={meta_info['train_minutes']}"
    ]
    subprocess.run(cmd, check=True)
    print(f"=> Đã lưu kết quả E5 vào {out_evidence}")

train_grpo_and_evaluate(best_adapter_name="E1", max_steps=60)


In [ ]:
# =========================================================================
# E5 (Bonus): Phân tích Reward Hacking và so sánh với SFT (3 điểm Rubric Bonus)
# =========================================================================
def analyze_grpo_vs_sft(sft_evidence="report/evidence/e1.json", grpo_evidence="report/evidence/e5.json"):
    if not Path(grpo_evidence).exists():
        print("Chưa có kết quả E5 để phân tích.")
        return
        
    sft = json.loads(Path(sft_evidence).read_text(encoding="utf-8"))
    grpo = json.loads(Path(grpo_evidence).read_text(encoding="utf-8"))
    
    print("=== SO SÁNH SFT vs GRPO ===")
    print(f"SFT  ({sft['name']}): F1={sft['micro']['f1']:.3f} | CI95=[{sft['f1_ci95'][0]:.3f}, {sft['f1_ci95'][1]:.3f}] | Mean IoU={sft['mean_iou']:.3f}")
    print(f"GRPO ({grpo['name']}): F1={grpo['micro']['f1']:.3f} | CI95=[{grpo['f1_ci95'][0]:.3f}, {grpo['f1_ci95'][1]:.3f}] | Mean IoU={grpo['mean_iou']:.3f}")
    
    # Kiểm tra dấu hiệu Reward Hacking (model lười chỉ trả [] hoặc box khổng lồ)
    grpo_preds = read_jsonl("work/e5.jsonl")
    empty_count = sum(1 for p in grpo_preds if p["raw"].strip() == "[]")
    empty_rate = empty_count / len(grpo_preds)
    print(f"\nTỷ lệ model trả về rỗng '[]': {empty_rate*100:.1f}% ({empty_count}/{len(grpo_preds)})")
    
    ci_sft = sft['f1_ci95']
    ci_grpo = grpo['f1_ci95']
    overlap = max(ci_sft[0], ci_grpo[0]) <= min(ci_sft[1], ci_grpo[1])
    print(f"Khoảng tin cậy CI95 giữa SFT và GRPO có chồng lấn không: {'CÓ (chưa khẳng định vượt trội thống kê)' if overlap else 'KHÔNG (chênh lệch có ý nghĩa thống kê)'}")

analyze_grpo_vs_sft()


In [ ]:
# Kiểm tra tổng thể bài nộp bằng check_submission.py
!python scripts/check_submission.py
